This notebook contains an implementation of Wagner's cross-entropy method adapted to finding a gonality savings graph. The goal of this is just to generate graphs from scratch.

We tried to make a faithful implementation of https://github.com/zawagner22/cross-entropy-for-combinatorics, replacing the SGD optimizer with an Adam optimizer with an exponential learning rate scheduller (performing a greedy search to tune hyperparameters). The same neural network architecture is used, though.

The other modification is that the graph cache consists of only UNIQUE graphs (up to isomorphism).

In [1]:
#### ENVIRONMENT SETUP

!pip install juliacall

#!/usr/bin/env python3
import os
import time
import math
import pickle
import random
import numpy as np
import networkx as nx
import scipy.stats as st


# ==========================================
# Julia Setup
# ==========================================
from juliacall import Main as jl

jl.seval("import Pkg")

# install packages (if not installed)
packages = ["ChipFiring", "Graphs", "TreeWidthSolver"]
for pkg in packages:
    jl.seval(f'if !haskey(Pkg.project().dependencies, "{pkg}") Pkg.add("{pkg}") end')

# initialize libraries
jl.seval("using ChipFiring")
jl.seval("using Graphs")
jl.seval("using TreeWidthSolver")

jl.seval("""
function fast_convert(np_array)
    return convert(Matrix{Int64}, np_array)
end
""")
fast_convert = jl.fast_convert

# --- PyTorch Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 10.6 MB/s eta 0:00:00
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliapkg/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliacall/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] WARNING: You have Julia 1.12.6 installed but 1.10.3 - 1.11 is required.
[juliapkg]   It is recommended that you upgrade Julia or install JuliaUp.
[juliapkg] Querying Julia versions from https://julialang-s3.julialang.org/bin/versions.json
[juliapkg] WARNING: About to install Julia 1.11.9 to /root/.julia/environments/pyjuliapkg/pyjuliapkg/install.
[juliapkg]   If you use juliapkg in more than one environment, you are likely to
[juliapkg]   have Julia installed in multiple locations. It is recommended to
[juliapkg]   install JuliaUp (https://github.com/JuliaLang/juliaup) or Julia
[juliapkg]   (https://julialang.org/downloads) yourself.
[juliapkg] Downloading Julia from htt

   Resolving package versions...
   Installed DataStructures ─ v0.19.5
   Installed ChipFiring ───── v0.4.1
   Installed ArnoldiMethod ── v0.4.0
   Installed Graphs ───────── v1.14.0
    Updating `~/.julia/environments/pyjuliapkg/Project.toml`
  [be096eff] + ChipFiring v0.4.1
    Updating `~/.julia/environments/pyjuliapkg/Manifest.toml`
  [ec485272] + ArnoldiMethod v0.4.0
  [be096eff] + ChipFiring v0.4.1
  [864edb3b] + DataStructures v0.19.5
  [86223c79] + Graphs v1.14.0
  [d25df0c9] + Inflate v0.1.5
  [699a6c99] + SimpleTraits v0.9.6
  [90137ffa] + StaticArrays v1.9.18
  [1e83bf80] + StaticArraysCore v1.4.4
  [10745b16] + Statistics v1.11.1
  [37e2e46d] + LinearAlgebra v1.11.0
  [2f01184e] + SparseArrays v1.11.0
  [e66e0078] + CompilerSupportLibraries_jll v1.1.1+0
  [4536629a] + OpenBLAS_jll v0.3.27+1
  [bea87d4a] + SuiteSparse_jll v7.7.0+0
  [8e850b90] + libblastrampoline_jll v5.11.0+0
Precompiling project...
    872.3 ms  ✓ StaticArraysCore
    965.6 ms  ✓ Inflate
   1398.8 ms  ✓ St

In [2]:
# ==========================================
# Hyperparameters & Constants
# ==========================================
N = 9   # Number of vertices in the graph
MYN = int(N * (N - 1) / 2)  # Length of the word (edges in complete graph)

EDGE_MULTIPLICITY = 4
MAX_EDGES_TO_CHECK = 21

INITIAL_LR = 0.02
LR_GAMMA = 0.99
N_SESSIONS = 200        # Number of new sessions per iteration
PERCENTILE = 90         # Top 100-X percentile we learn from
SUPER_PERCENTILE = 92   # Top 100-X percentile that survives to next iteration

# these are identical to the wagner paper
FIRST_LAYER_NEURONS = 128
SECOND_LAYER_NEURONS = 64
THIRD_LAYER_NEURONS = 4

OBSERVATION_SPACE = 2 * MYN
LEN_GAME = MYN
INF = 1000000
NUM_TRIALS = 5

# index computations
TRIU_I, TRIU_J = np.triu_indices(N, k=1)
device = torch.device("cpu")
print(f"Using compute device: {device} (Optimized for small MLP latency)")



Using compute device: cpu (Optimized for small MLP latency)


In [3]:
# this block contains the reward function we are using

score_cache = {}
found_gsgs = []



# random hyperparameters

alpha = 1000.0 + np.random.normal(0, 500.0)
beta = 20.0 + np.random.normal(0, 10.0)

# Reward function
def compute_gon_2_subdivision(g, gonality, n, num_nonzero_edges):
    if check_non_gsg(g, gonality, n, num_nonzero_edges):
        return gonality

    sub_g = jl.subdivide(g, 2)
    rank_to_check = gonality

    while rank_to_check > 0:
        if jl.compute_gonality(sub_g, min_d=rank_to_check, max_d=rank_to_check) == -1:
            return rank_to_check + 1
        rank_to_check -= 1

    return jl.compute_gonality(sub_g)

def check_non_gsg(g, gonality, n, num_edges):
    if n <= 5:
        return True
    if gonality <= 4:
        return True
    if num_edges > MAX_EDGES_TO_CHECK:
        return True
    genus = jl.compute_genus(g)
    if genus <= 5:
        return True
    return False

def is_connected_fast(edges, N):
    parent = list(range(N))

    def find(i):
        if parent[i] == i:
            return i
        parent[i] = find(parent[i])
        return parent[i]

    def union(i, j):
        root_i = find(i)
        root_j = find(j)
        if root_i != root_j:
            parent[root_i] = root_j

    for idx in range(len(edges)):
        if edges[idx] > 0:
            union(TRIU_I[idx], TRIU_J[idx])

    root = find(0)
    return all(find(i) == root for i in range(1, N))


def calcScore(state):
    global found_gsgs, score_cache
    edges = state[:MYN]
    state_key = tuple(edges)

    if state_key in score_cache:
        return score_cache[state_key]

    num_edges_total_weight = np.sum(edges)

    if num_edges_total_weight == 0 or not is_connected_fast(edges, N):
        score_cache[state_key] = 0.0
        return 0.0

    adj = np.zeros((N, N), dtype=int)
    adj[TRIU_I, TRIU_J] = edges
    adj[TRIU_J, TRIU_I] = edges

    try:
        jl_matrix = fast_convert(adj)
        graph = jl.ChipFiringGraph(jl_matrix)
        gon = int(jl.compute_gonality(graph))

        if gon < 5:
            score_cache[state_key] = float(gon)
            return float(gon)

        num_edges = np.sum(edges)
        gon2 = compute_gon_2_subdivision(graph, gon, N, num_edges)

        # CHECK FOR GONALITY SAVINGS GRAPH
        if gon > gon2:
            gsg_data = {
                'edges': edges.copy(),
                'gon': gon,
                'gon2': gon2,
                'v': N,
                'e': int(num_edges)
            }
            found_gsgs.append(gsg_data)

        treewidth = int(jl.exact_treewidth(jl.SimpleGraph(graph.adj_matrix)))
        reward = 50.0 + (alpha / num_edges_total_weight) - (beta * treewidth) + 100 * (gon - gon2) + np.random.normal(0, 0.1)

        score_cache[state_key] = reward
        return reward

    except Exception as e:
        score_cache[state_key] = 0.0
        return 0.0


In [4]:
# ==========================================
# PyTorch Model Definition
# ==========================================
class GraphGenerator(nn.Module):
    def __init__(self, obs_space, edge_multiplicity):
        super(GraphGenerator, self).__init__()
        self.fc1 = nn.Linear(obs_space, FIRST_LAYER_NEURONS)
        self.fc2 = nn.Linear(FIRST_LAYER_NEURONS, SECOND_LAYER_NEURONS)
        self.fc3 = nn.Linear(SECOND_LAYER_NEURONS, THIRD_LAYER_NEURONS)
        self.out = nn.Linear(THIRD_LAYER_NEURONS, edge_multiplicity + 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.out(x)


# ==========================================
# Helper Functions
# ==========================================
def generate_session(agent, n_sessions, verbose=False):
    agent.eval()
    states = np.zeros([n_sessions, OBSERVATION_SPACE, LEN_GAME], dtype=int)
    actions = np.zeros([n_sessions, LEN_GAME], dtype=int)
    state_next = np.zeros([n_sessions, OBSERVATION_SPACE], dtype=int)
    states[:, MYN, 0] = 1
    total_score = np.zeros([n_sessions])

    for step in range(1, LEN_GAME + 1):
        current_states = states[:, :, step - 1]
        current_states_t = torch.as_tensor(current_states, dtype=torch.float32)
        with torch.no_grad():
            logits = agent(current_states_t)
            prob = F.softmax(logits, dim=-1).numpy()

        cum_probs = np.cumsum(prob, axis=1)
        random_rolls = np.random.rand(n_sessions, 1)
        chosen_actions = np.argmax(random_rolls < cum_probs, axis=1)
        actions[:, step - 1] = chosen_actions

        state_next = current_states.copy()
        mask = chosen_actions > 0
        state_next[mask, step - 1] = chosen_actions[mask]
        state_next[:, MYN + step - 1] = 0

        if step < LEN_GAME:
            state_next[:, MYN + step] = 1

        terminal = step == LEN_GAME

        if terminal:
            for i in range(n_sessions):
                total_score[i] = calcScore(state_next[i])

        if not terminal:
            states[:, :, step] = state_next

    return states, actions, total_score

def select_elites(states_batch, actions_batch, rewards_batch, percentile=50):
    reward_threshold = np.percentile(rewards_batch, percentile)
    elite_indices = np.where(rewards_batch >= reward_threshold)[0]
    elite_states = states_batch[elite_indices].reshape(-1, OBSERVATION_SPACE)
    elite_actions = actions_batch[elite_indices].reshape(-1)
    return elite_states, elite_actions

def select_super_sessions(states_batch, actions_batch, rewards_batch, percentile=90):
    reward_threshold = np.percentile(rewards_batch, percentile)
    super_indices = np.where(rewards_batch >= reward_threshold)[0]
    return states_batch[super_indices], actions_batch[super_indices], rewards_batch[super_indices]

def filter_unique_isomorphic(states_batch, actions_batch, rewards_batch):
    """
    Filters a batch of graphs to ensure they are unique up to isomorphism.
    If duplicates exist, the instance with the highest reward is retained.
    """
    best_graphs = {}

    for idx, edges in enumerate(actions_batch):
        adj = np.zeros((N, N), dtype=int)
        adj[TRIU_I, TRIU_J] = edges
        adj[TRIU_J, TRIU_I] = edges

        nx_graph = nx.from_numpy_array(adj)
        # Use 'weight' as edge attribute to differentiate graphs by edge multiplicities
        graph_hash = nx.weisfeiler_lehman_graph_hash(nx_graph, edge_attr='weight')

        # Add if unseen, OR overwrite if the current graph has a better reward than the stored isomorphic match
        if graph_hash not in best_graphs or rewards_batch[idx] > best_graphs[graph_hash]['reward']:
            best_graphs[graph_hash] = {
                'idx': idx,
                'reward': rewards_batch[idx]
            }

    filtered_indices = [v['idx'] for v in best_graphs.values()]
    return states_batch[filtered_indices], actions_batch[filtered_indices], rewards_batch[filtered_indices]


The block below times the Wagner approach (no isomorphism checking) over NUM_TRIALS.

In [ ]:
found_gsgs = []
MAX_HOF_SIZE = 20 # Maximum number of unique historic elites to keep
NUM_TRIALS = globals().get('NUM_TRIALS', 5) # Fallback if hyperparameter cell wasn't run
TIME_LIMIT = 1200 # 400 seconds timeout per trial to find multiple graphs

# ==========================================
# Main Experiment Loop
# ==========================================
if __name__ == "__main__":
    trial_times = []

    print(f"--- Starting {NUM_TRIALS} Trials ---")

    for trial in range(NUM_TRIALS):

        # random hyperparameters
        alpha = 1000.0 + np.random.normal(0, 500.0)
        beta = 20.0 + np.random.normal(0, 10.0)
        print(f"\n--- Trial {trial + 1} / {NUM_TRIALS} --- alpha {alpha}, beta {beta}")



        score_cache.clear()
        found_gsgs = [] # Clear the found GSGs for this trial

        model = GraphGenerator(OBSERVATION_SPACE, EDGE_MULTIPLICITY).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=INITIAL_LR, weight_decay=0.01)
        scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=LR_GAMMA)
        loss_fn = nn.CrossEntropyLoss()

        # Rename to Hall of Fame (hof) to represent persistent memory
        hof_states = np.empty((0, LEN_GAME, OBSERVATION_SPACE), dtype=int)
        hof_actions = np.empty((0, LEN_GAME), dtype=int)
        hof_rewards = np.array([])

        trial_start_time = time.time()
        generation = 0

        while True:
            # Check for timeout
            if time.time() - trial_start_time > TIME_LIMIT:
                print(f"Time limit: Trial {trial + 1} exceeded {TIME_LIMIT} seconds limit.")
                break

            sessions = generate_session(model, N_SESSIONS, verbose=False)

            states_batch = np.transpose(sessions[0], axes=[0, 2, 1])
            actions_batch = sessions[1]
            rewards_batch = sessions[2]

            # Track Diversity
            unique_hashes = set()
            for edges in actions_batch:
                adj = np.zeros((N, N), dtype=int)
                adj[TRIU_I, TRIU_J] = edges
                adj[TRIU_J, TRIU_I] = edges
                nx_graph = nx.from_numpy_array(adj)
                unique_hashes.add(nx.weisfeiler_lehman_graph_hash(nx_graph, edge_attr='weight'))

            unique_graphs = len(unique_hashes)
            diversity_pct = (unique_graphs / N_SESSIONS) * 100

            # ==========================================
            # Hall of Fame Logic (Prevents Mode Collapse)
            # ==========================================

            # 1. Combine current batch with the historical Hall of Fame
            if len(hof_states) > 0:
                combined_states = np.vstack((states_batch, hof_states))
                combined_actions = np.vstack((actions_batch, hof_actions))
                combined_rewards = np.concatenate((rewards_batch, hof_rewards))
            else:
                combined_states = states_batch
                combined_actions = actions_batch
                combined_rewards = rewards_batch

            # 2. Filter the combined pool for unique isomorphisms
            unique_states, unique_actions, unique_rewards = filter_unique_isomorphic(
                combined_states, combined_actions, combined_rewards
            )

            # 3. Sort and keep only the absolute best unique graphs of all time
            sort_indices = np.argsort(unique_rewards)[::-1]
            hof_states = unique_states[sort_indices][:MAX_HOF_SIZE]
            hof_actions = unique_actions[sort_indices][:MAX_HOF_SIZE]
            hof_rewards = unique_rewards[sort_indices][:MAX_HOF_SIZE]

            # ==========================================
            # Training Data Prep
            # ==========================================

            # Get the elites from just the current batch to keep learning moving forward
            elite_states, elite_actions = select_elites(states_batch, actions_batch, rewards_batch, percentile=PERCENTILE)

            # Reshape back to 3D/2D to match hof_states and hof_actions
            elite_states = elite_states.reshape(-1, LEN_GAME, OBSERVATION_SPACE)
            elite_actions = elite_actions.reshape(-1, LEN_GAME)

            # Combine current generation elites with our historical Hall of Fame
            train_states = np.vstack((elite_states, hof_states))
            train_actions = np.vstack((elite_actions, hof_actions))

            # Train the network
            model.train()
            # Flatten back to 2D and 1D for the neural network training
            t_states = torch.as_tensor(train_states.reshape(-1, OBSERVATION_SPACE), dtype=torch.float32)
            t_actions = torch.as_tensor(train_actions.reshape(-1), dtype=torch.long)

            batch_size = 32
            dataset_size = len(t_states)
            indices = torch.randperm(dataset_size)

            for start_idx in range(0, dataset_size, batch_size):
                batch_idx = indices[start_idx : start_idx + batch_size]
                batch_states = t_states[batch_idx]
                batch_actions = t_actions[batch_idx]

                optimizer.zero_grad()
                logits = model(batch_states)
                loss = loss_fn(logits, batch_actions)
                loss.backward()
                optimizer.step()

            scheduler.step()

            if generation % 10 == 0:
                current_lr = scheduler.get_last_lr()[0]
                print(f"Gen {generation} | Max Reward: {np.max(rewards_batch):.2f} | "
                      f"LR: {current_lr:.5f} | "
                      f"Unique Graphs: {unique_graphs}/{N_SESSIONS} ({diversity_pct:.1f}%) | "
                      f"HoF Stored: {len(hof_states)} | GSGs Found: {len(found_gsgs)}")

            generation += 1

        trial_end_time = time.time()
        elapsed = trial_end_time - trial_start_time
        trial_times.append(elapsed)

        if found_gsgs:
            print(f"\nTrial {trial + 1} complete. Found {len(found_gsgs)} total GSGs.")
            unique_gsgs = {}
            for g_data in found_gsgs:
                adj = np.zeros((g_data['v'], g_data['v']), dtype=int)
                adj[TRIU_I, TRIU_J] = g_data['edges']
                adj[TRIU_J, TRIU_I] = g_data['edges']
                nx_graph = nx.from_numpy_array(adj)
                h = nx.weisfeiler_lehman_graph_hash(nx_graph, edge_attr='weight')
                if h not in unique_gsgs:
                    unique_gsgs[h] = g_data

            print(f"Filtering down to {len(unique_gsgs)} unique GSGs (up to isomorphism).")
            for idx, (h, g_data) in enumerate(unique_gsgs.items()):
                edges = g_data['edges']
                gon = g_data['gon']
                gon2 = g_data['gon2']
                v = g_data['v']
                e = g_data['e']
                filename = f"gsg_{trial+1}_{idx+1}_gon_{gon}_sigma2_{gon2}_v_{v}_e_{e}.txt"

                adj = np.zeros((v, v), dtype=int)
                adj[TRIU_I, TRIU_J] = edges
                adj[TRIU_J, TRIU_I] = edges

                with open(filename, 'w') as f:
                    for row in adj:
                        f.write(" ".join(map(str, row)) + "\n")
                print(f"Saved GSG to {filename}")
        else:
            print(f"Trial {trial + 1} finished without finding GSG (Time taken: {elapsed:.2f}s)")


--- Starting 5 Trials ---

--- Trial 1 / 5 --- alpha 764.6309195038707, beta 10.006711789962646
Gen 0 | Max Reward: 11.80 | LR: 0.01980 | Unique Graphs: 200/200 (100.0%) | HoF Stored: 20 | GSGs Found: 0
Gen 10 | Max Reward: 77.82 | LR: 0.01791 | Unique Graphs: 199/200 (99.5%) | HoF Stored: 20 | GSGs Found: 0
Gen 20 | Max Reward: 77.67 | LR: 0.01619 | Unique Graphs: 200/200 (100.0%) | HoF Stored: 20 | GSGs Found: 0
Gen 30 | Max Reward: 77.59 | LR: 0.01465 | Unique Graphs: 200/200 (100.0%) | HoF Stored: 20 | GSGs Found: 1
Gen 40 | Max Reward: 77.70 | LR: 0.01325 | Unique Graphs: 200/200 (100.0%) | HoF Stored: 20 | GSGs Found: 1
Gen 50 | Max Reward: 77.80 | LR: 0.01198 | Unique Graphs: 200/200 (100.0%) | HoF Stored: 20 | GSGs Found: 1
Gen 60 | Max Reward: 75.15 | LR: 0.01083 | Unique Graphs: 200/200 (100.0%) | HoF Stored: 20 | GSGs Found: 1
Gen 70 | Max Reward: 77.67 | LR: 0.00980 | Unique Graphs: 200/200 (100.0%) | HoF Stored: 20 | GSGs Found: 1
Gen 80 | Max Reward: 78.08 | LR: 0.00886 |